# Histórico – Tentativa inicial com ARIMA / ARIMAX / SARIMA / SARIMAX

Este notebook **documenta o percurso da pesquisa** e **não faz parte dos resultados do TCC**.

Na fase inicial, a desinformação foi tratada como variável exógena em modelos preditivos ("competição de modelos": ARIMA(0,1,1), ARIMAX(0,1,1), SARIMA(1,0,1)(1,1,1)[52] e SARIMAX), com treino em 2020–2022 e teste em 2023. A abordagem foi abandonada e substituída pela análise de CCF e causalidade de Granger (notebook 03).

**Origem:** células que estavam guardadas como texto no fim de `notebooks/03_correlacao_causalidade.ipynb`.  
**Incompleto:** as células anteriores (preparação de `y_treino`, `y_teste`, `X_treino`, `X_teste`, `auto_arima`, cálculo de métricas e o dicionário `resultados_modelos`) **não foram preservadas**; por isso o código abaixo **não roda sozinho**.

**Saídas gráficas da época:** `archive/outputs/01_…` a `06_…`.

In [ ]:
#-----------------------------------------------------------
# 5. Treinar e Avaliar ARIMA(0,1,1) (Melhor modelo não-sazonal)
#-----------------------------------------------------------
print("\n--- Treinando ARIMA(0,1,1) ---")

# Melhores parâmetros encontrados pelo auto_arima (apenas não-sazonais)
order_arima = (0, 1, 1)

# Criar e treinar o modelo
# Usar y_treino completo
modelo_arima = ARIMA(y_treino, order=order_arima)
resultado_arima = modelo_arima.fit()

print("\n--- Resumo do Modelo ARIMA(0,1,1) Treinado ---")
print(resultado_arima.summary())

#-----------------------------------------------------------
# 6. Fazer Previsões com ARIMA(0,1,1)
#-----------------------------------------------------------
print("\n--- Gerando Previsões ARIMA para o Período de Teste ---")

# Definir início e fim da previsão (índice do y_teste)
inicio_previsao = y_teste.index[0]
fim_previsao = y_teste.index[-1]

# Gerar as previsões
# Usar .predict() para previsões dentro da amostra (in-sample) ou fora da amostra (out-of-sample)
previsao_arima = resultado_arima.predict(start=inicio_previsao, end=fim_previsao)

print(f"Previsões geradas para {len(previsao_arima)} semanas.")

#-----------------------------------------------------------
# 7. Visualizar Previsões vs Dados Reais de Teste
#-----------------------------------------------------------
plt.figure(figsize=(14, 7))
# Mostra apenas o último ano do treino para melhor visualização
plt.plot(y_treino.loc['2022':], label='Treino (2022)')
plt.plot(y_teste, label='Real (Teste 2023)', color='blue')
plt.plot(previsao_arima, label='Previsão ARIMA(0,1,1) (2023)', color='orange', linestyle='--')
plt.title('Previsão ARIMA(0,1,1) vs Casos Reais (Teste)')
plt.xlabel('Data')
plt.ylabel('Casos Semanais')
plt.legend()
plt.grid(True)
plt.show()

#-----------------------------------------------------------
# 8. Calcular Métricas de Erro para ARIMA(0,1,1)
#-----------------------------------------------------------
# Garantir que y_teste e previsao_arima tenham o mesmo tamanho
if len(y_teste) == len(previsao_arima):
    rmse_arima = np.sqrt(mean_squared_error(y_teste, previsao_arima))
    mae_arima = mean_absolute_error(y_teste, previsao_arima)

    print(f"\n--- Métricas de Erro para o Modelo ARIMA(0,1,1) ---")
    print(f"RMSE: {rmse_arima:.4f}")
    print(f"MAE:  {mae_arima:.4f}")

    # Armazenar resultados para a tabela final
    # Inicializar o dicionário se ainda não existir
    if 'resultados_modelos' not in locals():
        resultados_modelos = {}

    resultados_modelos['ARIMA(0,1,1)'] = {'RMSE': rmse_arima, 'MAE': mae_arima, 'AIC': resultado_arima.aic} # Adicionar AIC

    print("\nAvaliação do ARIMA(0,1,1) concluída.")
else:
    print(f"\nERRO: Tamanho de y_teste ({len(y_teste)}) e previsão_arima ({len(previsao_arima)}) são diferentes.")
    print("Verifique as datas de início/fim da previsão.")

In [ ]:
#-----------------------------------------------------------
# 9. Treinar e Avaliar ARIMAX(0,1,1) (Incluindo Desinformação)
#-----------------------------------------------------------
print("\n--- Treinando ARIMAX(0,1,1) ---")

# Usar a mesma ordem do ARIMA base para comparação
order_arimax = (0, 1, 1)

# Verificar se X_treino tem valores antes de treinar
if X_treino.isnull().any().any():
    print("ALERTA: X_treino contém valores NaN. Preenchendo com 0.")
    X_treino.fillna(0, inplace=True)
if X_teste.isnull().any().any():
    print("ALERTA: X_teste contém valores NaN. Preenchendo com 0.")
    X_teste.fillna(0, inplace=True)


# Criar e treinar o modelo ARIMAX
# Inclui a variável exógena X_treino
modelo_arimax = ARIMA(y_treino, exog=X_treino, order=order_arimax)
resultado_arimax = modelo_arimax.fit()

print("\n--- Resumo do Modelo ARIMAX(0,1,1) Treinado ---")
print(resultado_arimax.summary())

#-----------------------------------------------------------
# 10. Fazer Previsões com ARIMAX(0,1,1)
#-----------------------------------------------------------
print("\n--- Gerando Previsões ARIMAX para o Período de Teste ---")

# Definir início e fim da previsão
inicio_previsao = y_teste.index[0]
fim_previsao = y_teste.index[-1]

# Gerar as previsões, FORNECENDO OS VALORES FUTUROS DA EXÓGENA (X_teste)
try:
    previsao_arimax = resultado_arimax.predict(start=inicio_previsao, end=fim_previsao, exog=X_teste)
    print(f"Previsões ARIMAX geradas para {len(previsao_arimax)} semanas.")

    #-----------------------------------------------------------
    # 11. Visualizar Previsões ARIMAX vs Dados Reais de Teste
    #-----------------------------------------------------------
    plt.figure(figsize=(14, 7))
    plt.plot(y_treino.loc['2022':], label='Treino (2022)')
    plt.plot(y_teste, label='Real (Teste 2023)', color='blue')
    plt.plot(previsao_arimax, label='Previsão ARIMAX(0,1,1) (2023)', color='green', linestyle='--')
    plt.title('Previsão ARIMAX(0,1,1) vs Casos Reais (Teste)')
    plt.xlabel('Data')
    plt.ylabel('Casos Semanais')
    plt.legend()
    plt.grid(True)
    plt.show()

    #-----------------------------------------------------------
    # 12. Calcular Métricas de Erro para ARIMAX(0,1,1)
    #-----------------------------------------------------------
    rmse_arimax = np.sqrt(mean_squared_error(y_teste, previsao_arimax))
    mae_arimax = mean_absolute_error(y_teste, previsao_arimax)

    print(f"\n--- Métricas de Erro para o Modelo ARIMAX(0,1,1) ---")
    print(f"RMSE: {rmse_arimax:.4f}")
    print(f"MAE:  {mae_arimax:.4f}")

    # Armazenar resultados
    resultados_modelos['ARIMAX(0,1,1)'] = {'RMSE': rmse_arimax, 'MAE': mae_arimax, 'AIC': resultado_arimax.aic} # Adicionar AIC

    print("\nAvaliação do ARIMAX(0,1,1) concluída.")

except ValueError as e:
    print(f"\nERRO ao gerar previsões ARIMAX: {e}")
    print("Verifique se X_teste cobre todo o período de previsão e não tem NaNs.")
    print(f"Índice y_teste: {y_teste.index.min()} a {y_teste.index.max()}")
    print(f"Índice X_teste: {X_teste.index.min()} a {X_teste.index.max()}")
    print(f"NaNs em X_teste: {X_teste.isnull().sum().sum()}")

In [ ]:
#-----------------------------------------------------------
# 13. Treinar e Avaliar SARIMA (Usando a melhor ordem do auto_arima)
#-----------------------------------------------------------
print("\n--- Treinando SARIMA(1,0,1)(1,1,1)[52] ---")

# Parâmetros encontrados pelo auto_arima como o melhor geral
order_sarima = (1, 0, 1)
seasonal_order_sarima = (1, 1, 1, 52)

# Criar e treinar o modelo SARIMA
modelo_sarima = ARIMA(y_treino, 
                      order=order_sarima, 
                      seasonal_order=seasonal_order_sarima,
                      enforce_stationarity=False, 
                      enforce_invertibility=False) 

# Treinar pode demorar um pouco mais devido à sazonalidade
resultado_sarima = modelo_sarima.fit()

print("\n--- Resumo do Modelo SARIMA Treinado ---")
print(resultado_sarima.summary())

#-----------------------------------------------------------
# 14. Fazer Previsões com SARIMA
#-----------------------------------------------------------
print("\n--- Gerando Previsões SARIMA para o Período de Teste ---")

inicio_previsao = y_teste.index[0]
fim_previsao = y_teste.index[-1]

previsao_sarima = resultado_sarima.predict(start=inicio_previsao, end=fim_previsao)
print(f"Previsões SARIMA geradas para {len(previsao_sarima)} semanas.")

#-----------------------------------------------------------
# 15. Visualizar Previsões SARIMA vs Dados Reais
#-----------------------------------------------------------
plt.figure(figsize=(14, 7))
plt.plot(y_treino.loc['2022':], label='Treino (2022)') 
plt.plot(y_teste, label='Real (Teste 2023)', color='blue')
plt.plot(previsao_sarima, label='Previsão SARIMA (2023)', color='purple', linestyle='--')
plt.title('Previsão SARIMA vs Casos Reais (Teste)')
plt.xlabel('Data')
plt.ylabel('Casos Semanais')
plt.legend()
plt.grid(True)
plt.show()

#-----------------------------------------------------------
# 16. Calcular Métricas de Erro para SARIMA
#-----------------------------------------------------------
rmse_sarima = np.sqrt(mean_squared_error(y_teste, previsao_sarima))
mae_sarima = mean_absolute_error(y_teste, previsao_sarima)

print(f"\n--- Métricas de Erro para o Modelo SARIMA ---")
print(f"RMSE: {rmse_sarima:.4f}")
print(f"MAE:  {mae_sarima:.4f}")

# Armazenar resultados
resultados_modelos['SARIMA(1,0,1)(1,1,1)[52]'] = {'RMSE': rmse_sarima, 'MAE': mae_sarima, 'AIC': resultado_sarima.aic}

print("\nAvaliação do SARIMA concluída.")

In [ ]:
#-----------------------------------------------------------
# 17. Treinar e Avaliar SARIMAX (Sazonal + Desinformação)
#-----------------------------------------------------------
print("\n--- Treinando SARIMAX(1,0,1)(1,1,1)[52] ---")

# Usar os mesmos parâmetros do SARIMA
order_sarimax = (1, 0, 1)
seasonal_order_sarimax = (1, 1, 1, 52)

# Criar e treinar o modelo SARIMAX
# Inclui a variável exógena X_treino
modelo_sarimax = ARIMA(y_treino, 
                       exog=X_treino, 
                       order=order_sarimax, 
                       seasonal_order=seasonal_order_sarimax,
                       enforce_stationarity=False, 
                       enforce_invertibility=False) 

# Treinar pode demorar um pouco
resultado_sarimax = modelo_sarimax.fit()

print("\n--- Resumo do Modelo SARIMAX Treinado ---")
print(resultado_sarimax.summary())

#-----------------------------------------------------------
# 18. Fazer Previsões com SARIMAX
#-----------------------------------------------------------
print("\n--- Gerando Previsões SARIMAX para o Período de Teste ---")

inicio_previsao = y_teste.index[0]
fim_previsao = y_teste.index[-1]

# Gerar previsões, FORNECENDO X_teste
try:
    previsao_sarimax = resultado_sarimax.predict(start=inicio_previsao, end=fim_previsao, exog=X_teste)
    print(f"Previsões SARIMAX geradas para {len(previsao_sarimax)} semanas.")

    #-----------------------------------------------------------
    # 19. Visualizar Previsões SARIMAX vs SARIMA vs Reais
    #-----------------------------------------------------------
    plt.figure(figsize=(14, 7))
    plt.plot(y_treino.loc['2022':], label='Treino (2022)') 
    plt.plot(y_teste, label='Real (Teste 2023)', color='blue', linewidth=2)
    # Plotar previsão SARIMA (anterior) para comparação
    plt.plot(previsao_sarima, label='Previsão SARIMA (ref)', color='purple', linestyle=':', alpha=0.7) 
    # Plotar previsão SARIMAX
    plt.plot(previsao_sarimax, label='Previsão SARIMAX (2023)', color='red', linestyle='--') 
    plt.title('Previsão SARIMAX vs SARIMA vs Casos Reais (Teste)')
    plt.xlabel('Data')
    plt.ylabel('Casos Semanais')
    plt.legend()
    plt.grid(True)
    plt.show()

    #-----------------------------------------------------------
    # 20. Calcular Métricas de Erro para SARIMAX
    #-----------------------------------------------------------
    rmse_sarimax = np.sqrt(mean_squared_error(y_teste, previsao_sarimax))
    mae_sarimax = mean_absolute_error(y_teste, previsao_sarimax)

    print(f"\n--- Métricas de Erro para o Modelo SARIMAX ---")
    print(f"RMSE: {rmse_sarimax:.4f}")
    print(f"MAE:  {mae_sarimax:.4f}")

    # Armazenar resultados
    resultados_modelos['SARIMAX(1,0,1)(1,1,1)[52]'] = {'RMSE': rmse_sarimax, 'MAE': mae_sarimax, 'AIC': resultado_sarimax.aic}

    print("\nAvaliação do SARIMAX concluída.")

except ValueError as e:
    print(f"\nERRO ao gerar previsões SARIMAX: {e}")
    print("Verifique X_teste.")

In [ ]:
#-----------------------------------------------------------
# 21. Tabela Final de Resultados da Competição
#-----------------------------------------------------------
print("\n--- Tabela Comparativa de Desempenho dos Modelos ---")

# Criar DataFrame a partir do dicionário de resultados
tabela_resultados = pd.DataFrame.from_dict(resultados_modelos, orient='index')
tabela_resultados = tabela_resultados.sort_values(by='RMSE') # Ordenar pelo menor RMSE

# Exibir a tabela formatada
print(tabela_resultados.round(4))

# Opcional: Salvar a tabela em CSV
# path_tabela = '../results/tables/tabela_comparativa_modelos.csv'
# tabela_resultados.to_csv(path_tabela)
# print(f"\nTabela de resultados salva em: {path_tabela}")

# Identificar o melhor modelo
melhor_modelo_nome = tabela_resultados.index[0]
melhor_rmse = tabela_resultados.iloc[0]['RMSE']
melhor_mae = tabela_resultados.iloc[0]['MAE']
melhor_aic = tabela_resultados.iloc[0]['AIC']

print(f"\nMelhor modelo encontrado: {melhor_modelo_nome}")
print(f"  RMSE: {melhor_rmse:.4f}")
print(f"  MAE:  {melhor_mae:.4f}")
print(f"  AIC (no treino): {melhor_aic:.3f}")

print("\n--- Fim da Competição de Modelos ---")